setup

In [ ]:

import pandas as pd
import numpy as np
import os

In [ ]:
BASE_DIR = os.path.dirname(os.getcwd())

DATA_DIR = os.path.join(
    BASE_DIR,
    "data"
)

RAW_DATA_DIR = os.path.join(
    DATA_DIR,
    "raw"
)

PROCESSED_DATA_DIR = os.path.join(
    DATA_DIR,
    "processed"
)

os.makedirs(
    PROCESSED_DATA_DIR,
    exist_ok=True
)

RAW_FILE = os.path.join(
    RAW_DATA_DIR,
    "ecommerce_data.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DATA_DIR,
    "ecommerce_cleaned.csv"
)

print("Base Directory:")
print(BASE_DIR)

print("\nRaw File:")
print(RAW_FILE)

print("\nOutput File:")
print(OUTPUT_FILE)

In [ ]:
df = pd.read_csv(RAW_FILE)

print("Dataset loaded successfully.")
print("Original Shape:", df.shape)

df.head()

In [ ]:
cleaned_df = df.copy()

In [ ]:
print("Original Columns:\n")

for column in cleaned_df.columns:
    print(column)

In [ ]:
cleaned_df.columns = (
    cleaned_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

print("Cleaned Columns:\n")

for column in cleaned_df.columns:
    print(column)

In [ ]:
rows_before = len(cleaned_df)

cleaned_df = (
    cleaned_df
    .dropna(how="all")
    .copy()
)

rows_removed = (
    rows_before
    - len(cleaned_df)
)

print(
    "Completely empty rows removed:",
    rows_removed
)

In [ ]:
duplicate_count = (
    cleaned_df
    .duplicated()
    .sum()
)

print(
    "Duplicate rows before cleaning:",
    duplicate_count
)

cleaned_df = (
    cleaned_df
    .drop_duplicates()
    .copy()
)

print(
    "Duplicate rows after cleaning:",
    cleaned_df.duplicated().sum()
)

In [ ]:
date_columns = []

for column in cleaned_df.columns:

    if (
        "date" in column.lower()
        or "time" in column.lower()
    ):
        date_columns.append(column)

print("Possible Date Columns:")
print(date_columns)

In [ ]:
for column in date_columns:

    cleaned_df[column] = pd.to_datetime(
        cleaned_df[column],
        errors="coerce"
    )

print("Date conversion completed.")

In [ ]:
text_columns = (
    cleaned_df
    .select_dtypes(
        include=["object", "string"]
    )
    .columns
    .tolist()
)

print("Text Columns:")
print(text_columns)

In [ ]:
for column in text_columns:

    cleaned_df[column] = (
        cleaned_df[column]
        .astype("string")
        .str.strip()
    )

print("Text cleaning completed.")

In [ ]:
for column in text_columns:

    cleaned_df[column] = (
        cleaned_df[column]
        .replace("", pd.NA)
    )

print("Empty strings replaced with missing values.")

In [ ]:
numeric_keywords = [
    "sales",
    "price",
    "profit",
    "cost",
    "discount",
    "quantity",
    "amount"
]

numeric_like_columns = []

for column in cleaned_df.columns:

    if any(
        keyword in column
        for keyword in numeric_keywords
    ):
        numeric_like_columns.append(column)

print("Numeric-like Columns:")
print(numeric_like_columns)

In [ ]:
for column in numeric_like_columns:

    cleaned_df[column] = pd.to_numeric(
        cleaned_df[column],
        errors="coerce"
    )

print("Numeric conversion completed.")

In [ ]:
for column in numeric_like_columns:

    cleaned_df[column] = pd.to_numeric(
        cleaned_df[column],
        errors="coerce"
    )

print("Numeric conversion completed.")

In [ ]:
id_columns = []

for column in cleaned_df.columns:

    if (
        column.endswith("_id")
        or "customer_id" in column
        or "order_id" in column
    ):
        id_columns.append(column)

print("ID Columns:")
print(id_columns)

In [ ]:
id_columns_with_missing = [
    column
    for column in id_columns
    if cleaned_df[column].isna().sum() > 0
]

rows_before = len(cleaned_df)

if id_columns_with_missing:

    cleaned_df = (
        cleaned_df
        .dropna(
            subset=id_columns_with_missing
        )
        .copy()
    )

rows_removed = (
    rows_before
    - len(cleaned_df)
)

print(
    "Rows removed because of missing IDs:",
    rows_removed
)

In [ ]:
numeric_columns = (
    cleaned_df
    .select_dtypes(
        include=np.number
    )
    .columns
    .tolist()
)

for column in numeric_columns:

    missing_count = (
        cleaned_df[column]
        .isna()
        .sum()
    )

    if missing_count > 0:

        median_value = (
            cleaned_df[column]
            .median()
        )

        cleaned_df[column] = (
            cleaned_df[column]
            .fillna(median_value)
        )

print(
    "Missing numeric values handled."
)

In [ ]:
categorical_columns = (
    cleaned_df
    .select_dtypes(
        include=["object", "string"]
    )
    .columns
    .tolist()
)

for column in categorical_columns:

    missing_count = (
        cleaned_df[column]
        .isna()
        .sum()
    )

    if missing_count > 0:

        cleaned_df[column] = (
            cleaned_df[column]
            .fillna("Unknown")
        )

print(
    "Missing categorical values handled."
)

In [ ]:
negative_report = []

for column in numeric_columns:

    negative_count = (
        cleaned_df[column] < 0
    ).sum()

    negative_report.append({
        "Column": column,
        "Negative Values": negative_count
    })

negative_report = pd.DataFrame(
    negative_report
)

negative_report

In [ ]:
non_negative_keywords = [
    "sales",
    "price",
    "cost",
    "quantity",
    "amount",
    "discount"
]

for column in numeric_columns:

    if any(
        keyword in column
        for keyword in non_negative_keywords
    ):

        invalid_count = (
            cleaned_df[column] < 0
        ).sum()

        if invalid_count > 0:

            print(
                f"{column}: "
                f"{invalid_count} invalid negative values found"
            )

            cleaned_df.loc[
                cleaned_df[column] < 0,
                column
            ] = np.nan

            cleaned_df[column] = (
                cleaned_df[column]
                .fillna(
                    cleaned_df[column].median()
                )
            )

print(
    "Invalid negative values handled."
)

In [ ]:
quantity_columns = [
    column
    for column in cleaned_df.columns
    if "quantity" in column
]

print("Quantity Columns:")
print(quantity_columns)

In [ ]:
for column in quantity_columns:

    invalid_count = (
        cleaned_df[column] <= 0
    ).sum()

    if invalid_count > 0:

        print(
            f"{column}: "
            f"{invalid_count} invalid rows removed"
        )

        cleaned_df = (
            cleaned_df[
                cleaned_df[column] > 0
            ]
            .copy()
        )

In [ ]:
discount_columns = [
    column
    for column in cleaned_df.columns
    if "discount" in column
]

for column in discount_columns:

    maximum_discount = (
        cleaned_df[column]
        .max()
    )

    if maximum_discount > 1:

        cleaned_df[column] = (
            cleaned_df[column] / 100
        )

print("Discount values standardized.")

In [ ]:
for column in date_columns:

    missing_dates = (
        cleaned_df[column]
        .isna()
        .sum()
    )

    print(
        f"{column}: "
        f"{missing_dates} missing/invalid dates"
    )

In [ ]:
if date_columns:

    rows_before = len(cleaned_df)

    cleaned_df = (
        cleaned_df
        .dropna(
            subset=date_columns
        )
        .copy()
    )

    rows_removed = (
        rows_before
        - len(cleaned_df)
    )

    print(
        "Rows removed due to invalid dates:",
        rows_removed
    )

In [ ]:
if date_columns:

    primary_date_column = date_columns[0]

    cleaned_df = (
        cleaned_df
        .sort_values(
            by=primary_date_column
        )
        .reset_index(
            drop=True
        )
    )

    print(
        "Dataset sorted by:",
        primary_date_column
    )

In [ ]:
final_missing_report = pd.DataFrame({
    "Missing Count": cleaned_df.isna().sum(),
    "Missing Percentage": (
        cleaned_df.isna().mean() * 100
    ).round(2)
})

final_missing_report = (
    final_missing_report[
        final_missing_report["Missing Count"] > 0
    ]
    .sort_values(
        "Missing Count",
        ascending=False
    )
)

final_missing_report

In [ ]:
final_duplicate_count = (
    cleaned_df
    .duplicated()
    .sum()
)

print(
    "Final Duplicate Rows:",
    final_duplicate_count
)

In [ ]:
print("=" * 70)
print("FINAL DATA CLEANING VALIDATION")
print("=" * 70)

print(
    "Original Shape:",
    df.shape
)

print(
    "Final Shape:",
    cleaned_df.shape
)

print(
    "Rows Removed:",
    len(df) - len(cleaned_df)
)

print(
    "Total Missing Values:",
    cleaned_df.isna().sum().sum()
)

print(
    "Duplicate Rows:",
    cleaned_df.duplicated().sum()
)

print("\nFinal Data Types:")
print(cleaned_df.dtypes)

print("=" * 70)

In [ ]:
cleaned_df.head()

In [ ]:
cleaned_df.describe(
    include="all"
)

In [ ]:
cleaned_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("=" * 70)
print("CLEANED DATASET SAVED SUCCESSFULLY")
print("=" * 70)

print(
    "File Location:"
)

print(OUTPUT_FILE)

print(
    "\nFinal Dataset Shape:",
    cleaned_df.shape
)

In [ ]:
verification_df = pd.read_csv(
    OUTPUT_FILE
)

print(
    "Saved Dataset Shape:",
    verification_df.shape
)

verification_df.head()

In [ ]:
print("=" * 70)

print(
    "02_DATA_CLEANING COMPLETED SUCCESSFULLY"
)

print("=" * 70)

print(
    "Input File:"
)

print(
    "data/raw/ecommerce_data.csv"
)

print("\nOutput File:")

print(
    "data/processed/ecommerce_cleaned.csv"
)

print("\nNext Pipeline:")

print(
    "03_sql_analysis"
)

print("=" * 70)